# Lab 03 — Embeddings & Memory Representation

*PyTorch · MemN2N · Guided challenges and solutions*

---
## Challenge 1 — Word Embeddings

### 1.1 — Create the embedding layer
- [ ] Create an `nn.Embedding` using the vocabulary size and embedding dimension.
- [ ] Configure the padding ID appropriately.
- [ ] Inspect the weight matrix shape.


In [86]:
import torch
import torch.nn as nn

tokens = torch.tensor([3, 6, 5, 4, 0])


vocab_size = 8
embedding_dim = 4
pad_id = 7

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=pad_id
)

output = embedding(tokens)

print(output)
print(output.shape)
print(embedding.weight.shape)

tensor([[ 0.5546,  0.2264, -0.4021,  0.4839],
        [ 0.3375,  0.3645, -0.2526, -0.7405],
        [ 1.6726,  1.4469, -0.6052, -0.2336],
        [-1.1422, -0.2498,  2.2319,  1.7842],
        [ 0.3020, -0.1005, -0.0671, -1.5779]], grad_fn=<EmbeddingBackward0>)
torch.Size([5, 4])
torch.Size([8, 4])


### 1.2 — Retrieve one token embedding
- [ ] Pass one token ID (`3`) through the existing embedding layer.
- [ ] Display its vector and shape `(4,)`.


In [87]:
token = torch.tensor(3)
print(embedding(token))
print(embedding(token).shape)

tensor([ 0.5546,  0.2264, -0.4021,  0.4839], grad_fn=<EmbeddingBackward0>)
torch.Size([4])


### 1.3 — Encode a batch of sentences
- [ ] Pass the two padded sentences through the same layer.
- [ ] Inspect the three dimensions.

In [88]:
tokens_batch = torch.tensor([
    [3, 6, 5, 4, 0],
    [1, 6, 5, 2, 7]
])
print(embedding(tokens_batch))
print(embedding(tokens_batch).shape)



tensor([[[ 0.5546,  0.2264, -0.4021,  0.4839],
         [ 0.3375,  0.3645, -0.2526, -0.7405],
         [ 1.6726,  1.4469, -0.6052, -0.2336],
         [-1.1422, -0.2498,  2.2319,  1.7842],
         [ 0.3020, -0.1005, -0.0671, -1.5779]],

        [[-0.9130,  1.9053,  0.7873, -0.1526],
         [ 0.3375,  0.3645, -0.2526, -0.7405],
         [ 1.6726,  1.4469, -0.6052, -0.2336],
         [-0.2208,  0.0833,  0.2183,  0.3111],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>)
torch.Size([2, 5, 4])


---
## Challenge 2 — Sentence Representation

**Objective:** Collapse the token dimension to obtain **one vector per sentence**.

### 2.1 — Bag-of-Words (sum)
- [ ] Sum word embeddings across the **token dimension**.
- [ ] Print the batch output shape `(2, 4)`.

**Concept:** Sum-based Bag-of-Words does not encode word order.


In [89]:
embedded_batch = embedding(tokens_batch)

sentence_vectors = embedded_batch.sum(dim=1)

print(sentence_vectors)
print(sentence_vectors.shape)

tensor([[ 1.7246,  1.6875,  0.9049, -0.2838],
        [ 0.8764,  3.7999,  0.1479, -0.8156]], grad_fn=<SumBackward1>)
torch.Size([2, 4])


**Sum Pooling** aggregates token embeddings into a single sentence vector by summing them element-wise.

### Mathematical Formula

$$
\mathbf{s} = \sum_{i=1}^{L} \mathbf{e}_i
$$

Where:
- $L$ → Number of tokens in the sentence.
- $\mathbf{e}_i$ → Embedding vector of token $i$.
- $\mathbf{s}$ → Resulting sentence vector.

- **Input:** `[batch_size, seq_len, embedding_dim]`
- **Output:** `[batch_size, embedding_dim]`

- `sum(dim=1)` → Sums token vectors **element-wise** along the sequence dimension.
- Sum pooling produces **one vector per sentence**.
- Sum pooling does **not preserve token order**.
- Padding embeddings contribute zero if initialized and kept at zero using `padding_idx`.
- Unlike mean pooling, sum pooling does not divide by the number of tokens.


### 2.2 — Apply a padding mask
- [ ] Build a mask with `1` for genuine tokens, `0` for `<PAD>`.
- [ ] Adapt mask dimensions so multiplication with embeddings works.
- [ ] Multiply embeddings by the mask, then aggregate.
- [ ] Confirm that padded positions do not contribute.

**Hint (conceptual):** Token embeddings have three dimensions; a 2D mask needs an extra final dimension for broadcasting.

**Expected final batch shape:** `(2, 4)`.

In [90]:
mask = tokens_batch != pad_id

mask = mask.unsqueeze(-1)

masked_embeddings = embedded_batch * mask

sentence_vectors = masked_embeddings.sum(dim=1)

print(sentence_vectors)

tensor([[ 1.7246,  1.6875,  0.9049, -0.2838],
        [ 0.8764,  3.7999,  0.1479, -0.8156]], grad_fn=<SumBackward1>)


### 2.3 — Sum vs. mean
- [ ] Compute a masked **mean** sentence vector.
- [ ] Divide by the number of actual tokens, not the padded sequence length.
- [ ] Compare results with the masked sum.


In [91]:
masked_sum = (embedded_batch * mask).sum(dim=1)

token_counts = mask.sum(dim=1)

sentence_mean = masked_sum / token_counts

print(sentence_mean)
print(sentence_mean.shape)

tensor([[ 0.3449,  0.3375,  0.1810, -0.0568],
        [ 0.2191,  0.9500,  0.0370, -0.2039]], grad_fn=<DivBackward0>)
torch.Size([2, 4])


**Mean Pooling** computes the average of token embeddings, excluding padding tokens using a mask.

### Mathematical Formula

$$
\mathbf{s}_{\text{mean}} = \frac{\sum_{i=1}^{L} m_i \mathbf{e}_i}{\sum_{i=1}^{L} m_i}
$$

Where:
- $L$ → Sequence length (including padding).
- $m_i$ → Mask value (`1` for real tokens, `0` for PAD).
- $\mathbf{e}_i$ → Embedding vector of token $i$.
- $\mathbf{s}_{\text{mean}}$ → Resulting sentence vector.


- Mean pooling excludes padding and averages only valid token embeddings.
- Mean pooling does **not preserve token order**.


---
## Challenge 3 — MemN2N Embeddings (`A`, `B`, `C`)

| Component | Input | Role | Symbol |
|---|---|---|---|
| **A** | Story sentence | Input memory representation | `mᵢ` |
| **B** | Question | Query representation | `u` |
| **C** | Story sentence | Output memory representation | `cᵢ` |

### 3.1 — Create three embedding layers
- [ ] Define separate PyTorch embedding layers `A`, `B`, and `C`.
- [ ] Use the same vocabulary size and embedding dimension.
- [ ] Inspect their weight shapes.

In [92]:
A = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=pad_id
)
B = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=pad_id
)
C = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=pad_id
)

print("A:", A.weight.shape)
print("B:", B.weight.shape)
print("C:", C.weight.shape)

A: torch.Size([8, 4])
B: torch.Size([8, 4])
C: torch.Size([8, 4])


### 3.2 — Encode story memories with A and C
- [ ] Pass the same sentence IDs through **A** and **C**.
- [ ] Aggregate token vectors into sentence vectors.
- [ ] Name the outputs `m` and `c`.
- [ ] Check their dimensions `(2, 4)`.


In [93]:
story = torch.tensor([
    [3, 6, 5, 4, 0],
    [1, 6, 5, 2, 7]
])


m = A(story)
c = C(story)

m = m.sum(dim=1)
c = c.sum(dim=1)

print(m.shape , c.shape)


torch.Size([2, 4]) torch.Size([2, 4])


### 3.3 — Encode a question with B
- [ ] Apply **B** and aggregate to a question vector `u`.
- [ ] Inspect its four components (shape `(4,)`).


In [94]:
question = torch.tensor([3, 6, 5])

u = B(question).sum(dim=0)  # dim=0 cuz it sums across tokens (rows)
print(u)

tensor([ 1.9477, -0.4373, -0.1499, -2.1701], grad_fn=<SumBackward1>)


---
## Challenge 4 — Position Encoding

**Objective:** Understand why pure Bag-of-Words ignores word order and how MemN2N can incorporate token positions.

### 4.1 — Observe Bag-of-Words limitations
- [ ] Create two token-ID sequences containing the **same words in a different order**.
- [ ] Compute their sum-based sentence vectors using the same embedding layer.
- [ ] Compare the representations.


In [95]:
sentence1 = torch.tensor([3, 6, 5, 4, 0])
sentence2 = torch.tensor([0, 4, 5, 6, 3])
s1 = A(sentence1).sum(dim=0)
s2 = A(sentence2).sum(dim=0)
print(torch.allclose(s1, s2))



True


### 4.2 — Add position-aware weighting
- [ ] Create a position-weight matrix with shape `(sequence_length, embedding_dim)`.
- [ ] Apply position weights **element-wise** to token embeddings before summing.
- [ ] Preserve padding behavior.
- [ ] Compare the representations of the reordered sentences.

In [96]:
position_weights = torch.tensor([
    [0.1, 0.2, 0.3, 0.4],
    [0.2, 0.3, 0.4, 0.5],
    [0.3, 0.4, 0.5, 0.6],
    [0.4, 0.5, 0.6, 0.7],
    [0.5, 0.6, 0.7, 0.8]
])

token_weighted = position_weights * A(sentence1)

s1 = token_weighted.sum(dim=0)
s2 = (position_weights * A(sentence2)).sum(dim=0)

print(s1.shape) 
print(s2.shape) 

print(torch.allclose(s1, s2))

torch.Size([4])
torch.Size([4])
False


**Position Weighting** assigns weights to token embeddings based on their position in a sequence.

**Mathematical Formula:**

$$
\mathbf{s} = \sum_{i=1}^{L} p_i \mathbf{e}_i
$$

- $p_i$ → Weight assigned to position $i$.
- $\mathbf{e}_i$ → Embedding vector of token $i$.
- $L$ → Sequence length.

*In this exercise, each position has a 4D weight vector; multiplication is element-wise. These weights are illustrative.*


---
## Challenge 5 — Batched Story Memory Representation

**Objective:** Process multiple stories while keeping the story, memory-slot and embedding dimensions separate.

### 5.1 — Batch multiple stories
- [ ] Build a toy batch containing **2 stories**, each with **2 sentences** of length `5`.
- [ ] Apply A and C and aggregate across the **word dimension only**.
- [ ] Confirm all shapes.

| Tensor | Expected shape | Meaning |
|---|---|---|
| Story token IDs | `(2, 2, 5)` | batch, memory slots, tokens |
| Word embeddings | `(2, 2, 5, 4)` | + embedding dimension |
| Memory vectors `m`, `c` | `(2, 2, 4)` | batch, slots, embedding dimension |
| Question vectors `u` | `(2, 4)` | batch, embedding dimension |

**Note:** Memory slots correspond to sentences. Attention over slots (`dim=1`) belongs to Lab 04, not this notebook.


In [97]:
stories = torch.tensor([
    [
        [3, 6, 5, 4, 0],
        [1, 6, 5, 2, 7]
    ],
    [
        [1, 6, 5, 4, 2],
        [3, 6, 5, 0, 7]
    ]
])

questions = torch.tensor([
    [3, 6, 5],
    [1, 6, 5]
])

a_embeddings = A(stories)
c_embeddings = C(stories)

m = a_embeddings.sum(dim=2)
c = c_embeddings.sum(dim=2)

u = B(questions).sum(dim=1)

print("Stories:", stories.shape)
print("Word embeddings:", a_embeddings.shape)
print("Input memories (m):", m.shape)
print("Output memories (c):", c.shape)
print("Question vectors (u):", u.shape)

Stories: torch.Size([2, 2, 5])
Word embeddings: torch.Size([2, 2, 5, 4])
Input memories (m): torch.Size([2, 2, 4])
Output memories (c): torch.Size([2, 2, 4])
Question vectors (u): torch.Size([2, 4])


---
## Lab 03 — Completion checklist

- [ ] I can distinguish token IDs from embedding vectors.
- [ ] I understand the meaning of `(batch, tokens, embedding_dim)`.
- [ ] I can convert word vectors into one sentence vector.
- [ ] I can handle padding when aggregating embeddings.
- [ ] I understand why MemN2N has separate `A`, `B`, and `C` tables.
- [ ] I can represent a story as multiple memory slots.
- [ ] I can explain the shapes of `m`, `c`, and `u`.

**Next:** Lab 04 — Attention & Weighted Memory Retrieval (`u · mᵢ` → softmax → weighted sum of `cᵢ`).
